In [ ]:
import numpy as np 
import pandas as pd
from sklearn.decomposition import PCA
import plot_utils as plu
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plot_utils as plu

## PCA

### Compute PCs

#### Using scikit-learn regular PCA
- combined_PCA: all datasets are merged and PCA is computed once
- coupled_PCA: for each generated dataset, merge with the real dataset and compute PCA
- independent_PCA: PCA on each dataset separately (not even merged with real)


For each type of PCA, output:
- separated scatterplots
- superpose scatterplot (generated overlayed on top of gray Real)
- density plots of the PC scores
- computing distance between real and generated scores (eg wasserstein distance)

In [ ]:
print()
print("#################### PCA ####################")
print("- Computing and plotting PCA...")

In [ ]:
scorelist = []
ncomp=6 # change to compute more PCs

In [ ]:
if matching_SNPs:
    methodname="Combined PCA"
    method='combined_PCA'
    print(f'    - Computing and plotting {methodname} ...')
    pca = PCA(n_components=ncomp)
    pcs = pca.fit_transform(
        np.concatenate(list(datasets.values()))
        )
    pcdf = pd.DataFrame(pcs, columns=["PC{}".format(x+1) for x in np.arange(pcs.shape[1])] )
    pcdf["label"] = sampleinfo.label.astype('category')
    plu.plotPCAallfigs(pcdf, methodname, orderedCat=infiles.keys(), outDir=outDir, colpal=colpal)
    plt.show()

    # Distances in this common basis (Reviewer 2, comment 3).
    # The coupled PCA below refits a separate basis per method, so its distances
    # live in different spaces with different retained axes and scaling -- a valid
    # within-method diagnostic, but not comparable across methods. Scoring in the
    # combined basis instead (the one Figures 2 and S2 are drawn in) puts every
    # dataset in the same space, so the values can be compared directly.
    # Both are kept: they are appended to the same scorelist and end up in
    # scores_all_PCA.csv distinguished by the 'method' column.
    k = plu.computePCAdist(pcdf, method, outDir, stat='wasserstein')
    scorelist.append(k)
    print('Scorelist-wasserstein', k)

    k = plu.computePCAdist(pcdf, method, outDir, stat='wasserstein2D', reg=2*1e-3)
    scorelist.append(k)
    print('Scorelist-wasserstein2D', k)

In [ ]:

methodname="Coupled PCA"
method='coupled_PCA'
print(f'Computing {methodname} ...')
nReal = datasets['Real'].shape[0]
pcdf=pd.DataFrame()
for cat in categ:
    pca = PCA(n_components=ncomp)
    pcs = pca.fit_transform(
        np.concatenate([datasets['Real'],datasets[cat]])
    ) # PCA on combined Real + cat individuals
    #df = pd.DataFrame(pcs[nReal:,:], columns=["PC{}".format(x+1) for x in np.arange(pcs.shape[1])]) #keep only pc values for individuals in cat
    #df.insert(ncomp,'label',cat)
    df = pd.DataFrame(pcs, columns=["PC{}".format(x+1) for x in np.arange(pcs.shape[1])]) #keep only pc values for individuals in cat
    df['label']=np.concatenate([['Real']*nReal,[cat]*datasets[cat].shape[0]])
    df['coupled_with'] = cat
    pcdf = pd.concat( [pcdf, df], ignore_index=True)
    
# plot all PCA figures and compute KS 
plu.plotPCAallfigs(pcdf, methodname, orderedCat=infiles.keys(), outDir=outDir, colpal=colpal)
plt.show()
    

k = plu.computePCAdist(pcdf,method,outDir,stat='wasserstein')
scorelist.append(k)
print('Scorelist-wasserstein', k)
plt.show()

k = plu.computePCAdist(pcdf, method, outDir, stat='wasserstein2D',reg=2*1e-3) 
scorelist.append(k)
print('Scorelist-wasserstein2D', k)

In [ ]:
if allchecks or not matching_SNPs: 
    # compute if not matching_SNPs even if not allchecks 
    # because combined or coupled PCA are not possible in this case
    methodname="Independent PCA"
    method='independent_PCA'
    print(f'Computing {methodname} ...')
    pcdf=pd.DataFrame()
    for cat in categ:
        pca = PCA(n_components=ncomp)
        pcs = pca.fit_transform(datasets[cat])
        df = pd.DataFrame(pcs, columns=["PC{}".format(x+1) for x in np.arange(pcs.shape[1])])
        df.insert(ncomp,'label',cat)
        pcdf = pd.concat( [pcdf, df], ignore_index=True)

    # plot all PCA figures  
    plu.plotPCAallfigs(pcdf, methodname, orderedCat=infiles.keys(), outDir=outDir, colpal=colpal)


In [ ]:

scores_pca = pd.concat(scorelist, sort=False)
scores_pca.to_csv(outDir+'scores_all_PCA.csv')

# average scores (distances) accross PC axes
sc_sum_over_PCs = scores_pca.groupby(['method','stat','label'])['statistic'].sum()
sc_mean_over_PCs = scores_pca.groupby(['method','stat','label'])['statistic'].mean()
print(sc_mean_over_PCs)

In [ ]:
# Supplementary table: PCA distances in the COMMON (combined) basis.
# Companion to Table S1, whose PCA rows come from the coupled per-method fit.
# Writes this dataset's rows, then rebuilds the LaTeX table from whichever
# datasets have been run so far (run the notebook once per DATA to get both).
import glob

ORDER = ['Truth', 'Indep', 'Markov', 'HMM', 'WGAN', 'RBM', 'GPC']
LATEX = {'Truth': r'\textsc{Truth}', 'Indep': r'\indep', 'Markov': r'\markov',
         'HMM': r'\hmm', 'WGAN': r'\gan', 'RBM': r'\rbm', 'GPC': r'\method'}
PCPAIRS = ['1-2', '3-4', '5-6']

_c = scores_pca[(scores_pca.method == 'combined_PCA') &
                (scores_pca.stat == 'wasserstein2D')]
if len(_c):
    _c.to_csv(outDir + 'pca_common_basis.csv', index=False)
    print('- common-basis PCA distances (wasserstein2D):')
    print(_c.pivot_table(index='PC', columns='label', values='statistic'))

    # side-by-side with the coupled values, to show what the basis change does
    _k = scores_pca[(scores_pca.method == 'coupled_PCA') &
                    (scores_pca.stat == 'wasserstein2D')]
    if len(_k):
        print('\n- coupled (Table S1) for comparison:')
        print(_k.pivot_table(index='PC', columns='label', values='statistic'))

# collect every dataset that has been run (FIGS/<DATA>/<settings>/pca_common_basis.csv)
_found = {}
for _f in sorted(glob.glob(os.path.join(outDir, '..', '..', '*', '*', 'pca_common_basis.csv'))):
    _found[os.path.normpath(_f).split(os.sep)[-3]] = pd.read_csv(_f)

if _found:
    _L = [r'\begin{table}[h]', r'\centering', r'\small',
          r'\begin{tabular}{c|c|C{43pt}R{43pt}R{43pt}R{43pt}R{43pt}R{43pt}R{43pt}}',
          r'\toprule', r'\textbf{Dataset} &',
          '& ' + ' & '.join(LATEX[m] for m in ORDER) + r' \\', r'\midrule']
    for _i, (_ds, _df) in enumerate(sorted(_found.items())):
        _L.append(r'\multirow{3}{*}{\textbf{' + _ds + r'}}')
        for _pc in PCPAIRS:
            _sel = _df[_df.PC == _pc].set_index('label')['statistic']
            _vals = [float(_sel.get(m, float('nan'))) for m in ORDER]
            # bold the best generative model (exclude Truth, the training set)
            _best = min(range(1, len(_vals)), key=lambda j: _vals[j])
            _cells = [(r'\textbf{' + f'{v:.4f}' + '}') if j == _best else f'{v:.4f}'
                      for j, v in enumerate(_vals)]
            _L.append(f'& PCA{_pc} & ' + ' & '.join(_cells) + r' \\')
        if _i < len(_found) - 1:
            _L.append(r'\midrule')
    _L += [r'\bottomrule', r'\end{tabular}',
           r'    \caption{\textbf{PCA distances in a common basis.} Wasserstein 2D '
           r'(Sinkhorn, $\epsilon=2\times10^{-3}$) distances between the PCA '
           r'representations of real (test set) versus generated individuals. A single '
           r'PCA is fit once on the union of all eight datasets (training set, test set '
           r'and the six sets of artificial genomes) and every dataset is projected into '
           r'that same space, which is the space shown in Figures~\ref{fig:pca} and '
           r'\ref{fig:pca-all}. Unlike Table~\ref{tab:global-distance}, where a separate '
           r'PCA is fit per method, these values are directly comparable across methods. '
           r'Truth represents the training set. Bolded values indicate the best among all '
           r'compared models.}',
           r'\label{tab:pca-common-basis}', r'\end{table}']
    _tex = os.path.join(outDir, '..', '..', 'pca_common_basis_table.tex')
    with open(_tex, 'w') as _fh:
        _fh.write('\n'.join(_L) + '\n')
    print(f"\n- LaTeX table ({', '.join(sorted(_found))}) -> {os.path.normpath(_tex)}")
    print('\n'.join(_L))

In [ ]:
print("#################### PCA DONE ####################")